In [1]:
import os

In [2]:
%pwd

'f:\\Files\\DS&ML\\FareFinder\\Exp'

In [3]:
os.chdir('../')
%pwd

'f:\\Files\\DS&ML\\FareFinder'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_id:str
    local_data_file:Path
    unzip_dir:Path

In [5]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_id=config.source_id,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [7]:
import urllib.request as request
import gdown
import zipfile
from mlproject import logger
from mlproject.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            gdown.download(id=self.config.source_id, output=self.config.local_data_file, quiet=False)
            print(f"[INFO] Downloaded file: {self.config.local_data_file}")
        else:
            print(f"[INFO] File already exists: {self.config.local_data_file} ({get_size(Path(self.config.local_data_file))})")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            print(f"[INFO] Extracted files to: {unzip_path}")


In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    
except Exception as e:
    raise e

[2025-05-20 00:01:40,160: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-05-20 00:01:40,167: INFO: common: yaml file: params.yaml loaded successfully]
[2025-05-20 00:01:40,179: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-05-20 00:01:40,183: INFO: common: created directory at: artifacts]
[2025-05-20 00:01:40,184: INFO: common: created directory at: artifacts/data_ingestion]


[INFO] File already exists: artifacts/data_ingestion/data.zip (~ 2977 KB)
[INFO] Extracted files to: artifacts/data_ingestion
